# Week 1 Day 5 Lab 2 - Harness Engineering

In [2]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain.agents.middleware import TodoListMiddleware
from langchain_openai import ChatOpenAI

## Connect to OpenRouter

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if openrouter_api_key:
    print("OpenRouter API key loaded successfully.")
else:
    print("No OpenRouter API key found.")



OpenRouter API key loaded successfully.


## Part 1: The Claims Desk

In [4]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files. """
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False, held_out=False):
    """ Return a list of claims, and the handlers' decisions on the keyed by claim id.
    
    held_out=True gives he twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set."""
    claims =load('evals/held_out_claims.json' if held_out else 'claims.json')
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [5]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim['claim_id'], claim['summary'])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [6]:
DESK = Path ('lab2_desk')

def make_desk(claims):
    """ Build a fresh claims desk with the claims to decide and everything deeded to decide them. """
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ['claims', 'policies', 'customers', 'decisions']:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / 'claims' / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2),
        encoding='utf-8')
    for policy in load('policies.json'):
        (DESK / 'policies' / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2),
        encoding='utf-8')
    for customer in load('customers.json'):
        (DESK / 'customers' / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2),
        encoding='utf-8')
    for name in ['policy_wording.md', 'claims_manual.md', 'company.json']:
        shutil.copy(DATA / name, DESK / name)

make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

## Part 2: Setting Up The Harness

In [7]:
INSTRUCTIONS = """ 
You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
-policy_wording.md is the policy wording and claims_manual.md is the claims handling manual.
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """ 
You are a claims handler at Undercover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the  required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
then reply with one line: the claim id, the decision and the payout.
"""

In [8]:
llm = ChatOpenAI(
    model="deepseek/deepseek-v4.1-flash",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort='low',
    timeout=120,
    max_retries=5
)

In [9]:
claim_handler = {
    'name': 'claim-handler',
    'description': 'Decides on claim and writes its decision file. Give it the claim id.',
    'system_prompt': HANDLER_INSTRUCTIONS
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware()],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True)
)

## Part 3: Working the queue

In [10]:
def describe(tool_call):
    """ A short line about one tool call."""
    args = tool_call['args']
    if 'file_path' in args or 'path' in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call['name'] =='task':
        return f"task: {args.get('description','').splitlines()[0][:80]}"
    if tool_call['name'] == 'write_todos':
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call['name']

In [11]:
cost = 0
calls = 0

for namespace, update in agent.stream(
{'messages': [{
    'role': 'user',
    'content': 'Please work the queue'
}]},
stream_mode='updates',
subgraphs=True,
config={'recursion_limit': 200},
):
    if 'model' in update:
        indent= "     " if namespace else ""
        for message in update['model']['messages']:
            calls += 1
            cost += message.response_metadata['token_usage']['cost']
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())
print(f"\nFinished after {calls} model calls for ${cost:.4f}")

ls /
I'll start by exploring the folder structure to understand what I'm working with.
ls /claims/
ls /policies/
ls /customers/
ls /decisions/
read_file /company.json
read_file /claims_manual.md
read_file /policy_wording.md
read_file /claims_manual.md
read_file /policy_wording.md
read_file /claims/CLM-1007.json
read_file /claims/CLM-1015.json
read_file /claims/CLM-1017.json
read_file /claims/CLM-1020.json
read_file /claims/CLM-1021.json
Now let me read all the claims in the queue.
read_file /policies/POL-2007.json
read_file /policies/POL-2015.json
read_file /policies/POL-2017.json
read_file /policies/POL-2020.json
read_file /policies/POL-2021.json
read_file /customers/CUS-3007.json
read_file /customers/CUS-3015.json
read_file /customers/CUS-3017.json
read_file /customers/CUS-3020.json
read_file /customers/CUS-3021.json
write_todos: Review queue, policy wording,  (completed), Decide CLM-1007 (phone liquid  (in_progress), Decide CLM-1015 (trip interrup (in_progress), Decide CLM-1017 (bac

## Part 4 How It Did

In [12]:
def is_right(answer, label):
    """ True when the decision matches the handler's and the payout is within a dollar. """
    return answer.get('decision') == label['decision'] and abs(float(answer.get('payout', 0)) - label['payout']) <= 1

rows= []
for claim_id, label in labels.items():
    path = DESK / 'decisions' / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'decision': 'missing', 'payout': 0}
    rows.append({
        'claim': claim_id,
        'decision': answer.get('decision'),
        'payout': answer.get('payout'),
        "handler's decision": label['decision'],
        "handler's payout": label['payout'],
        'right': is_right(answer, label)
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

4 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50.0,approve,50.0,True
1,CLM-1015,decline,0.0,decline,0.0,True
2,CLM-1017,refer,0.0,refer,0.0,True
3,CLM-1020,approve,240.0,approve,240.0,True
4,CLM-1021,approve,954.0,approve,825.0,False


In [13]:
display(Markdown((DESK / 'handover.md').read_text(encoding='utf-8')))

# Handover — Claims Queue

All five claims in the queue have been decided. Decision records are in `/decisions/`.
Wording version 2026.1, manual version 2026.2.

## Summary

| Claim | Type | Plan | Decision | Payout | Notes |
|---|---|---|---|---|---|
| CLM-1007 | Baggage (phone water damage) | Economy | Approve | $50.00 | Accidentally damaged phone; per-item limit caps at $150, less $100 deductible. |
| CLM-1015 | Trip interruption | First | Decline | $0.00 | Reason not covered under s.2 (left child at home). |
| CLM-1017 | Baggage (snatch theft) | Economy | Refer | $0.00 (est. $420 if docs supplied) | Missing police report and proof of value for sunglasses. |
| CLM-1020 | Baggage (monkey took sunglasses) | First | Approve | $240.00 | Wild-animal rule = accidental loss; guide letter is sufficient evidence. |
| CLM-1021 | Baggage (snatch theft) | Premium + Electronics Plus | Approve | $954.00 | Console not "electronics" under add-on, capped at $300; $50 deductible. |

**No claim requires payment sign-off** — all approved payouts are at or below the $1,000 authority limit.

## Things to watch

- **CLM-1017 — document request outstanding.** Referred (not declined) for missing documents. A 30-day request has been issued naming: (1) the Carabinieri police report for the theft, and (2) proof of value for the sunglasses claimed at $130. The claim stays open and the customer should be told the exact documents and how to send them. If supplied, it computes to $420: backpack $90, tablet capped at $150, jacket capped at $150, sunglasses $130 = $520, less $100 deductible. (Note: the story says the jacket was in the bag but only the backpack, tablet and jacket receipts — sunglasses value is the gap.)
- **CLM-1021 — console classification.** The handheld game console was deliberately NOT treated as "electronics" under Electronics Plus (that definition covers phones, tablets, laptops, cameras/lenses, e-readers, smartwatches only), so it was capped at the Premium $300 per-item limit rather than $1,200. If you disagree with this reading, revisit before payment.
- **CLM-1020 — wild animal.** Treated as accidental loss per the manual's interpretation note; no police report needed, guide's written statement accepted. This is intentional despite the "theft by monkey" framing.
- **CLM-1015 — decline reasoning.** Payout is $0; the decline rests on s.2's closed list of covered reasons. Warm but firm letter; no lecture.

## Data handling
All five customers have clean prior-claims histories (no repeat-claim/SIU referral needed on this basis). No conflicts between story and documents were found in any claim, so no SIU referral was raised.
